In [1]:
import sys

print(sys.executable)

/Users/ipekcoskunuzer/Desktop/mercari-shoe-resale-analysis/.venv-1/bin/python


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Pandas version:", pd.__version__)
print("Everything imported successfully!")

Pandas version: 3.0.3
Everything imported successfully!


In [4]:
from pathlib import Path

data_path = Path("../data/raw/train.tsv")

print("File exists:", data_path.exists())
print("File location:", data_path)

File exists: True
File location: ../data/raw/train.tsv


In [5]:
df = pd.read_csv(
    data_path,
    sep="\t",
    low_memory=False
)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1482535, 8)


,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description
0,0,MLB Cincinnati Reds T Shirt Size XL,3,Men/Tops/T-shirts,NaN,10.0,1,No description yet
1,1,Razer BlackWidow Chroma Keyboard,3,Electronics/Computers & Tablets/Components & P...,Razer,52.0,0,This keyboard is in great condition and works ...
2,2,AVA-VIV Blouse,1,Women/Tops & Blouses/Blouse,Target,10.0,1,Adorable top with a hint of lace and a key hol...
3,3,Leather Horse Statues,1,Home/Home Décor/Home Décor Accents,NaN,35.0,1,New with tags. Leather horses. Retail for [rm]...
4,4,24K GOLD plated rose,1,Women/Jewelry/Necklaces,NaN,44.0,0,Complete with certificate of authenticity


In [6]:
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1482535 entries, 0 to 1482534
Data columns (total 8 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   train_id           1482535 non-null  int64  
 1   name               1482535 non-null  str    
 2   item_condition_id  1482535 non-null  int64  
 3   category_name      1476208 non-null  str    
 4   brand_name         849853 non-null   str    
 5   price              1482535 non-null  float64
 6   shipping           1482535 non-null  int64  
 7   item_description   1482529 non-null  str    
dtypes: float64(1), int64(3), str(4)
memory usage: 384.4 MB


train_id                  0
name                      0
item_condition_id         0
category_name          6327
brand_name           632682
price                     0
shipping                  0
item_description          6
dtype: int64

In [7]:
women_shoe_categories = (
    df.loc[
        df["category_name"]
        .fillna("")
        .str.startswith("Women/Shoes"),
        "category_name"
    ]
    .value_counts()
)

women_shoe_categories

category_name
Women/Shoes/Boots                 18864
Women/Shoes/Sandals               14662
Women/Shoes/Athletic              12662
Women/Shoes/Fashion Sneakers      10164
Women/Shoes/Pumps                  7454
Women/Shoes/Flats                  5422
Women/Shoes/Loafers & Slip-Ons     4509
Women/Shoes/Slippers               1822
Women/Shoes/Mules & Clogs           929
Women/Shoes/Other                   739
Women/Shoes/Oxfords                 175
Women/Shoes/Work & Safety           138
Women/Shoes/Outdoor                 114
Name: count, dtype: int64

In [8]:
women_shoes = df[
    df["category_name"]
    .fillna("")
    .str.startswith("Women/Shoes")
].copy()

women_shoes["shoe_type"] = (
    women_shoes["category_name"]
    .str.split("/")
    .str[2]
)

brand_coverage = (
    women_shoes
    .groupby("shoe_type")
    .agg(
        total_listings=("train_id", "size"),
        listings_with_brand=("brand_name", "count")
    )
)

brand_coverage["brand_coverage_percent"] = (
    brand_coverage["listings_with_brand"]
    / brand_coverage["total_listings"]
    * 100
)

brand_coverage.sort_values(
    "total_listings",
    ascending=False
)

,total_listings,listings_with_brand,brand_coverage_percent
shoe_type,,,
Boots,18864,11470,60.803647
Sandals,14662,9992,68.148956
Athletic,12662,11793,93.136945
Fashion Sneakers,10164,8963,88.183786
Pumps,7454,4432,59.458009
Flats,5422,3694,68.129841
Loafers & Slip-Ons,4509,2865,63.539587
Slippers,1822,1351,74.149286
Mules & Clogs,929,412,44.348762


In [9]:
selected_categories = [
    "Boots",
    "Sandals",
    "Athletic",
    "Fashion Sneakers",
    "Pumps",
    "Flats",
    "Loafers & Slip-Ons"
]

In [10]:
condition_summary = (
    women_shoes
    .groupby("item_condition_id")
    .agg(
        listing_count=("train_id", "size"),
        median_price=("price", "median"),
        average_price=("price", "mean")
    )
)

condition_summary

,listing_count,median_price,average_price
item_condition_id,,,
1,17800,33.0,46.931236
2,21076,26.0,36.261909
3,34646,24.0,31.852681
4,3988,16.0,22.057673
5,144,12.0,17.250000


In [11]:
brand_counts = (
    women_shoes["brand_name"]
    .value_counts()
)

print("Number of unique brands:", women_shoes["brand_name"].nunique())
brand_counts.head(30)

Number of unique brands: 775


brand_name
Nike                 9053
UGG Australia        3568
Converse             3224
VANS                 2481
Adidas               1978
Tory Burch           1746
Steve Madden         1714
PINK                 1586
Michael Kors         1474
Coach                1434
Charlotte Russe      1093
TOMS                 1008
Birkenstock           914
PUMA                  886
FOREVER 21            867
SKECHERS              677
Hunter                592
American Eagle        524
Mossimo               508
Victoria's Secret     500
Dr. Martens           469
Timberland            451
Nine West             440
New Balance           438
Crocs                 430
ALDO                  372
GUESS                 359
Sperrys               351
Jessica Simpson       345
ASICS                 337
Name: count, dtype: int64

In [12]:
threshold_summary = pd.Series({
    "Brands with at least 50 listings": (brand_counts >= 50).sum(),
    "Brands with at least 100 listings": (brand_counts >= 100).sum(),
    "Brands with at least 250 listings": (brand_counts >= 250).sum(),
    "Brands with at least 500 listings": (brand_counts >= 500).sum()
})

threshold_summary

Brands with at least 50 listings     120
Brands with at least 100 listings     76
Brands with at least 250 listings     41
Brands with at least 500 listings     20
dtype: int64

In [15]:
women_shoes["price"].describe()

count    77654.000000
mean        35.975610
std         35.725738
min          0.000000
25%         16.000000
50%         26.000000
75%         41.000000
max        770.000000
Name: price, dtype: float64

In [16]:
women_shoes["price"].quantile(
    [0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1.00]
)

0.00      0.0
0.01      7.0
0.05     10.0
0.25     16.0
0.50     26.0
0.75     41.0
0.95     94.0
0.99    176.0
1.00    770.0
Name: price, dtype: float64

In [17]:
price_checks = pd.Series({
    "Zero-dollar listings": (women_shoes["price"] == 0).sum(),
    "Listings below $5": (women_shoes["price"] < 5).sum(),
    "Listings above $200": (women_shoes["price"] > 200).sum(),
    "Listings above $500": (women_shoes["price"] > 500).sum()
})

price_checks

Zero-dollar listings     50
Listings below $5       155
Listings above $200     475
Listings above $500      29
dtype: int64